[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/03_consensus.ipynb)

# Paper 3: what the Delphi rule does when nobody is watching

A modified Delphi rule keeps an item when the median is high, the spread is small and
enough raters call it relevant. Reasonable on its face. Simulate it against items whose
true endorsement you know, and one of the three criteria turns out to be doing nearly all
the work, in the wrong direction.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: about 45 seconds. Full run: about two minutes.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here takes about two minutes.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## The rule and the rating model

On the integer scale $1,\dots,9$ an item is retained when

$$\text{median} \ge 7, \qquad \mathrm{IQR} \le \delta, \qquad \text{I-CVI} \ge 0.78,$$

where the content-validity index is the share of raters giving at least 7. Quartiles are
the linear-interpolation order statistics, `numpy.percentile` with `method="linear"`, type
7 in the usual taxonomy. That convention is part of the rule, and later on it turns out to
be load bearing.

Ratings are generated as $x = \operatorname{clip}(\operatorname{round}(N(\mu,\sigma)),1,9)$.
An item's true endorsement $\pi$ is the probability that a single rating is at least 7 on
that rounded scale, so $\mu$ is obtained by inverting the endorsement of the rounded
variable, not the continuous one. Inverting the continuous version would leave a request
for $\pi = 0.90$ realising 0.956, and every error rate computed from it would describe a
different item.

In [ ]:
banner()

from hpotential.core.consensus import (
    endorsement_probability, iqr_lemma_violations, mu_for_endorsement,
    operating_characteristics, retention_masks, simulate_ratings,
)

SIGMA = 1.2
mu_good = mu_for_endorsement(0.90, SIGMA)
mu_marginal = mu_for_endorsement(0.65, SIGMA)

print(f"a well-endorsed item, pi = 0.90 -> mu = {mu_good:.4f}")
print(f"a marginal item,      pi = 0.65 -> mu = {mu_marginal:.4f}")
print()
print("check the inverse round-trips on the rounded scale:")
print(f"  endorsement at mu = {mu_good:.4f}: {endorsement_probability(mu_good, SIGMA):.6f}")

# Now solve the same request on the continuous scale and see what it really delivers.
from scipy.stats import norm

mu_continuous = 7.0 - SIGMA * norm.ppf(0.10)          # P(N(mu, sigma) >= 7) = 0.90
realised = endorsement_probability(mu_continuous, SIGMA)
print()
print(f"the continuous solution      mu = {mu_continuous:.4f}")
print(f"what it realises once ratings are rounded: {realised:.4f}, not 0.90")
print()
print("Half a scale point of mu, and the item under test is a different item. Every")
print("error rate below would describe that other item instead.")

## The retention curve

For a genuinely good item, $\pi = 0.90$, how often does the rule drop it? Intuition says
a bigger panel is a safer panel. Run it.

In [ ]:
banner()

import time

n_sim = 20000 if FULL_RUN else 5000
panel_sizes = (12, 20, 30, 45, 60)

start = time.time()
rng = np.random.default_rng(20260815)
rows = []
for n_panel in panel_sizes:
    good = operating_characteristics(n_panel, 0.90, SIGMA, rng, n_sim, iqr_threshold=1.5)
    marginal = operating_characteristics(n_panel, 0.65, SIGMA, rng, n_sim, iqr_threshold=1.5)
    rows.append({"panel": n_panel,
                 "false drop (pi=0.90)": good["p_drop"],
                 "false keep (pi=0.65)": marginal["p_retain"],
                 "median bound": good["median_fails"],
                 "IQR bound": good["iqr_fails"],
                 "CVI bound": good["cvi_fails"]})
oc = pd.DataFrame(rows)
print(f"{n_sim} panels per cell, {time.time() - start:.1f} s")
oc.round(4)

In [ ]:
banner()

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))

ax = axes[0]
ax.plot(oc["panel"], oc["false drop (pi=0.90)"], "o-", ms=5, lw=1.5, color=series_color(1),
        label=r"false drop, $\pi$ = 0.90")
ax.plot(oc["panel"], oc["false keep (pi=0.65)"], "s-", ms=5, lw=1.5, color=series_color(0),
        label=r"false keep, $\pi$ = 0.65")
ax.set_xlabel("panel size"); ax.set_ylabel("rate"); ax.set_ylim(0, 1)
ax.set_title("A bigger panel drops good items more often", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
bottom = np.zeros(len(oc))
for i, col in enumerate(["median bound", "IQR bound", "CVI bound"]):
    ax.bar(oc["panel"].astype(str), oc[col], 0.6, bottom=bottom, color=series_color(i), label=col)
    bottom = bottom + oc[col].to_numpy()
ax.set_xlabel("panel size"); ax.set_ylabel("share of dropped panels failing each test")
ax.set_title("Which criterion did the dropping", fontsize=10)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print("The shares sum past one because several criteria can fail together. That is")
print("the point of reporting all three rather than the first one that fired.")

**The diagnosis.** The median criterion essentially never bites. The content-validity
index bites less and less as the panel grows. The dispersion limit bites on nearly every
drop, and it gets worse with panel size, because the sample interquartile range converges
on a population value above 1.5. The rule is not becoming more reliable with a larger
panel. It is becoming more reliably wrong.

## Is it the threshold or the criterion?

Hold everything else fixed and sweep the dispersion limit $\delta$ over four candidates,
while panel disagreement $\sigma$ grows. If the problem were the value 1.5, some other
value would fix it.

In [ ]:
banner()

sigmas = np.round(np.arange(0.5, 2.5, 0.1), 3) if FULL_RUN else np.round(np.arange(0.5, 2.5, 0.2), 3)
candidates = (1.0, 1.5, 2.0, 2.5)
diag_sim = 20000 if FULL_RUN else 4000

start = time.time()
rng = np.random.default_rng(20260815)
diagnostic = {}
for threshold in candidates:
    diagnostic[threshold] = [
        operating_characteristics(30, 0.90, float(s), rng, diag_sim, iqr_threshold=threshold)["p_drop"]
        for s in sigmas
    ]
print(f"{len(candidates) * len(sigmas)} cells in {time.time() - start:.1f} s")

fig, ax = plt.subplots(figsize=(6.6, 4.0))
for i, threshold in enumerate(candidates):
    ax.plot(sigmas, diagnostic[threshold], "o-", ms=4, lw=1.4, color=series_color(i),
            label=rf"$\delta$ = {threshold}")
ax.set_xlabel(r"panel disagreement $\sigma$")
ax.set_ylabel(r"false-drop rate for an item with $\pi$ = 0.90")
ax.set_ylim(0, 1)
ax.set_title("Panel of 30. Two of these thresholds are alive; two are not.", fontsize=10)
ax.legend(fontsize=8)
plt.show()

pd.DataFrame(diagnostic, index=pd.Index(sigmas, name="sigma")).round(3)

## The dispersion lemma, and why $\delta \ge 2$ is inert

The 2.0 and 2.5 curves are flat and low. That is not luck.

**Lemma.** On the integer scale $1,\dots,9$ with relevance cut 7 and linear-interpolation
quartiles, an item whose I-CVI is strictly above 0.75 has $Q_1 \ge 7$. The scale maximum is
9, so $Q_3 \le 9$, and therefore $\mathrm{IQR} \le 2$. A limit $\delta \ge 2$ can never
bind, and the rule collapses to the median and the endorsement.

*Proof.* $Q_{0.25}$ interpolates between the order statistics at position $1 + 0.25(n-1)$.
If strictly more than three quarters of ratings are at least 7, then fewer than a quarter
fall below 7, so both order statistics bracketing that position are at least 7, hence
$Q_{0.25} \ge 7$. With $\kappa = 0.78 > 0.75$ the hypothesis holds. $\square$

Both hypotheses are load bearing. At exactly $\kappa = 0.75$ the panel $(1,7,8,8)$ has
I-CVI 0.75, median 7.5 and IQR 2.5. And under a different quantile convention the lemma is
simply false: under the Weibull rule the panel $(1,7,9,9,9)$ has I-CVI 0.8 and IQR 5.

You do not have to take this on trust. Every panel of every size up to 10 can be
enumerated, because the statistics do not care about rating order, so sorted multisets
suffice.

In [ ]:
banner()

max_panel = 10 if FULL_RUN else 8
start = time.time()
linear = iqr_lemma_violations(max_panel=max_panel, cvi_min=0.78, iqr_bound=2.0, method="linear")
weibull = iqr_lemma_violations(max_panel=max_panel, cvi_min=0.78, iqr_bound=2.0, method="weibull")
print(f"exhaustive enumeration up to panel size {max_panel}, {time.time() - start:.1f} s")
print()
for label, out in [("linear interpolation (type 7)", linear), ("Weibull rule (type 6)", weibull)]:
    print(f"{label}")
    print(f"  panels enumerated        {out['n_panels_checked']}")
    print(f"  passing I-CVI >= 0.78    {out['n_passing_cvi']}")
    print(f"  IQR > 2 among those      {out['violations']}")
    print(f"  largest IQR among those  {out['max_iqr_among_passing']}   at {out['worst_panel']}")
    print()
print("Zero violations under the convention the rule names, and hundreds under one it")
print("does not. A protocol that adopts this lemma has to name its quantile rule, not")
print("only its thresholds.")

In [ ]:
banner()

# The worst case is reachable, not hypothetical. Check the boundary panel by hand.
worst = np.array([[7, 7, 9, 9]], dtype=float)
masks = retention_masks(worst, median_min=7.0, iqr_threshold=2.0, cvi_min=0.78)
print(f"panel (7, 7, 9, 9): median {masks['median'][0]}, IQR {masks['iqr'][0]}, "
      f"I-CVI {masks['cvi'][0]}")
print(f"retained at delta = 2.0: {bool(masks['retain'][0])}")

edge = np.array([[1, 7, 8, 8]], dtype=float)
m2 = retention_masks(edge, median_min=7.0, iqr_threshold=2.0, cvi_min=0.75)
print()
print(f"panel (1, 7, 8, 8) at the excluded boundary kappa = 0.75:")
print(f"  median {m2['median'][0]}, IQR {m2['iqr'][0]}, I-CVI {m2['cvi'][0]}")
print(f"  IQR exceeds 2: {m2['iqr'][0] > 2.0}   <- why the lemma needs kappa > 0.75 strictly")

## Interactive: rediscover which criterion binds

Three sliders: panel size, panel disagreement, and the dispersion limit. The output is the
false-drop rate for an item that genuinely deserves to stay, and the share of drops each
criterion is responsible for.

Push the dispersion limit to 2.0 and watch the false-drop rate collapse and the IQR share
go to zero. Then push the panel size up at a limit of 1.5 and watch the false-drop rate
climb. Those two moves are the paper.

In [ ]:
banner()

widget_sim = 3000


def rule_panel(panel=30, sigma=1.2, iqr_limit=1.5):
    rng_w = np.random.default_rng(11)
    good = operating_characteristics(int(panel), 0.90, float(sigma), rng_w, widget_sim,
                                     iqr_threshold=float(iqr_limit))
    marginal = operating_characteristics(int(panel), 0.65, float(sigma), rng_w, widget_sim,
                                         iqr_threshold=float(iqr_limit))

    fig, ax = plt.subplots(figsize=(6.0, 3.2))
    names = ["median", "IQR", "I-CVI"]
    shares = [good["median_fails"], good["iqr_fails"], good["cvi_fails"]]
    ax.barh(names, shares, color=[series_color(i) for i in range(3)])
    ax.set_xlim(0, 1.0)
    ax.set_xlabel("share of wrongly dropped panels on which this criterion failed")
    ax.set_title(f"panel {int(panel)}, sigma {sigma:.1f}, IQR limit {iqr_limit:.1f}: "
                 f"false drop {good['p_drop']:.3f}", fontsize=10)
    plt.show()

    print(f"false-drop rate for a good item (pi = 0.90)   {good['p_drop']:.4f}")
    print(f"false-keep rate for a weak item (pi = 0.65)   {marginal['p_retain']:.4f}")
    print(f"binding shares: median {good['median_fails']:.3f}, "
          f"IQR {good['iqr_fails']:.3f}, I-CVI {good['cvi_fails']:.3f}")
    if iqr_limit >= 2.0:
        print()
        print("At a limit of 2.0 or more the dispersion criterion cannot bind at all,")
        print("by the lemma above. Its share is zero because it is inert, not because")
        print("it agrees with the others.")


explore(
    rule_panel,
    controls=dict(panel=(6, 60, 2), sigma=(0.4, 2.6, 0.1), iqr_limit=(0.5, 3.0, 0.5)),
    defaults=dict(panel=30, sigma=1.2, iqr_limit=1.5),
)

## The stakeholder split

Panels are mixed. Suppose patients rate an item highly and clinicians do not. As the
clinician endorsement falls, how long does the item survive the pooled rule? And does the
answer depend on how many of each sit on the panel?

In [ ]:
banner()

split_sim = 20000 if FULL_RUN else 3000
clinician_pi = np.linspace(0.05, 0.98, 41 if FULL_RUN else 21)
patient_mu = 8.3

start = time.time()
rng = np.random.default_rng(20260815)
curves = {}
for n_patients, n_clinicians in ((6, 24), (12, 18), (15, 15)):
    probs = []
    for pi_clin in clinician_pi:
        mu_clin = mu_for_endorsement(float(pi_clin), SIGMA)
        patients = simulate_ratings(rng, patient_mu, SIGMA, (split_sim, n_patients))
        clinicians = simulate_ratings(rng, mu_clin, SIGMA, (split_sim, n_clinicians))
        masks = retention_masks(np.concatenate([patients, clinicians], axis=1),
                                median_min=7.0, iqr_threshold=1.5, cvi_min=0.78)
        probs.append(float(masks["retain"].mean()))
    curves[f"{n_patients} patients : {n_clinicians} clinicians"] = np.array(probs)
print(f"{time.time() - start:.1f} s")

fig, ax = plt.subplots(figsize=(6.6, 4.0))
for i, (label, probs) in enumerate(curves.items()):
    ax.plot(clinician_pi, probs, lw=1.6, color=series_color(i), label=label)
ax.axhline(0.80, ls="--", lw=0.9, color=INK)
ax.set_xlabel(r"clinician endorsement $\pi$")
ax.set_ylabel("probability the item is retained")
ax.set_ylim(0, 1)
ax.set_title(r"An item patients rate at $\mu$ = 8.3, as clinicians cool on it", fontsize=10)
ax.legend(fontsize=8)
plt.show()

for label, probs in curves.items():
    at_half = float(np.interp(0.50, clinician_pi, probs))
    need = float(np.interp(0.80, probs, clinician_pi)) if probs.max() >= 0.80 else float("nan")
    print(f"{label:<32} P(retained | clinician pi = 0.50) = {at_half:.3f}, "
          f"clinician pi needed for 0.80 = {need:.3f}")

**What that shows.** Even a 15:15 split gives a patient-endorsed item roughly a one in
twelve chance of surviving when clinicians are split down the middle, and clinicians have
to reach about 0.97 endorsement before the item is safe. Weighting the panel toward
patients barely helps, because a mixed panel raises dispersion, and dispersion is the
criterion doing the dropping.

---

## What this does not establish

**These are properties of a decision rule, not of a panel.** No Delphi has been run. No
expert has rated anything. The error rates describe what the rule does to items whose true
endorsement is known by construction, which is exactly the thing you never know in
practice.

**The rating model is an assumption.** Rounded, clipped normal ratings with one common
$\sigma$. Real panels cluster, argue and drift between rounds. A different rating model
gives different rates, though the mechanism, an order-statistic criterion tightening as
$n$ grows, does not depend on normality.

**The lemma is about a rule and a quantile convention together.** It is true under linear
interpolation and false under Weibull, Hazen and the median-unbiased rules. It is not a
statement about the thresholds alone.

**"Dispersion binds on 99 per cent of drops" is conditional on a drop happening.** It is
not the probability that dispersion is wrong.

**A reduced run does not reproduce the published rates.** The lemma enumeration is exact
either way; only its panel-size ceiling changes.